# Aula 05 — Laboratório Pandas II — Agrupamentos e Joins

**Disciplina:** Análise e Ciência de Dados
**Curso:** Especialização em Ciência de Dados — IFRN
**Professor:** Plácido Antônio de Souza Neto

---

## Antes de começar

1. Renomeie este arquivo para `aula05_SEUNOME.ipynb`.
2. Mantenha a pasta `dados/` ao lado do notebook, com os arquivos
   `matriculas_ifrn_2026.csv` e `campi_ifrn.xlsx` — **os mesmos da Aula 04**.
3. Responda **na célula indicada**, sem apagar os enunciados.
4. Execute tudo do zero antes de entregar (`Kernel → Restart & Run All`)
   e confira se não há erros.
5. Entregue o `.ipynb` **já executado** (com as saídas visíveis) no Moodle.

> Os dados são **sintéticos**, os mesmos da Aula 04: nomes de campi e cursos
> são reais, mas matrículas, nomes de estudantes e notas foram gerados
> aleatoriamente. Nenhuma conclusão sobre o IFRN pode ser tirada deles.

## Preparação padrão (execute antes de tudo)

```python
import numpy as np
import pandas as pd

df = pd.read_csv("dados/matriculas_ifrn_2026.csv", sep=";", decimal=",",
                 encoding="utf-8", dtype={"matricula": "string"})
df["campus"] = df["campus"].str.strip()
df = df.drop_duplicates().reset_index(drop=True)

campi = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="campi")
cursos = pd.read_excel("dados/campi_ifrn.xlsx", sheet_name="cursos")
```

## Dicionário de dados

Igual ao da Aula 04 — veja a apostila se precisar relembrar as colunas de
`matriculas_ifrn_2026.csv`. Novidade desta aula:

| Tabela | Colunas |
|---|---|
| `campi` | `campus`, `cidade`, `regiao`, `ano_criacao`, `num_docentes` |
| `cursos` | `curso`, `eixo_tecnologico`, `nivel`, `carga_horaria` |


In [75]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Recapitulando a leitura e limpeza da Aula 04

In [76]:
import numpy as np
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/aula05_laboratorio_dados/dados/matriculas_ifrn_2026.csv", sep=";", decimal=",",
                 encoding="utf-8", dtype={"matricula": "string"})

df["campus"] = df["campus"].str.strip()
df = df.drop_duplicates().reset_index(drop=True)

campi = pd.read_excel("/content/drive/MyDrive/aula05_laboratorio_dados/dados/campi_ifrn.xlsx", sheet_name="campi")
cursos = pd.read_excel("/content/drive/MyDrive/aula05_laboratorio_dados/dados/campi_ifrn.xlsx", sheet_name="cursos")

print("Estudantes sem nota final:", len(df[df["nota_final"].isna()]))
print(f"matrículas: {df.shape} | campi: {campi.shape} | cursos: {cursos.shape}")

Estudantes sem nota final: 41
matrículas: (480, 12) | campi: (12, 5) | cursos: (10, 4)


## Questão 1 — `groupby` simples (1,0 ponto)

Calcule, a partir de `df` já limpo (ver preparação padrão):

a) a nota média por `turno`;
b) o número de estudantes (`size`) por `modalidade`;
c) explique, em uma frase, a diferença entre usar `.size()` e usar
   `.count()` sobre a coluna `nota_final` dentro de um `groupby`.

In [77]:
print("## a) a nota média por turno \n")
print(df.groupby("turno")["nota_final"].mean())
print("\n----------\n")

print("## b) o número de estudantes (size) por modalidade \n")
print(df.groupby("modalidade").size())
print("\n----------\n")

## a) a nota média por turno 

turno
Matutino      70.895683
Noturno       66.760606
Vespertino    71.616667
Name: nota_final, dtype: float64

----------

## b) o número de estudantes (size) por modalidade 

modalidade
EaD           107
Presencial    373
dtype: int64

----------



_Sua resposta escrita:_

c) explique, em uma frase, a diferença entre usar .size() e usar .count() sobre a coluna nota_final dentro de um groupby.

A diferença é que .size() conta todas linhas do grupo mesmo que ausentes enquanto .count() apenas os não ausentes.


## Questão 2 — `groupby` com múltiplas colunas (1,0 ponto)

a) Agrupe por `campus` e `turno` simultaneamente e calcule a nota média.
b) Que tipo de índice o resultado tem? Use `type(resultado.index)` para
   confirmar.
c) Converta o resultado para uma tabela "larga" (campus nas linhas, turno
   nas colunas) usando `unstack()`.

In [78]:
print("## a) Agrupe por campus e turno simultaneamente e calcule a nota média \n")
resultado = df.groupby(["campus", "turno"])["nota_final"].mean()
print(resultado)
print("\n----------\n")

print("## b) Que tipo de índice o resultado tem? Use type(resultado.index) para confirmar. \n")
print(type(resultado.index))
print("\n----------\n")

print("## c) Converta o resultado para uma tabela \"larga\" (campus nas linhas, turno nas colunas) usando unstack(). \n")
print(resultado.unstack())
print("\n----------\n")


## a) Agrupe por campus e turno simultaneamente e calcule a nota média 

campus                   turno     
Apodi                    Matutino      69.890909
                         Noturno       71.975000
                         Vespertino    73.737500
Caicó                    Matutino      67.950000
                         Noturno       72.370000
                         Vespertino    67.900000
Currais Novos            Matutino      75.280000
                         Noturno       66.790000
                         Vespertino    78.525000
João Câmara              Matutino      70.212500
                         Noturno       61.793750
                         Vespertino    60.971429
Macau                    Matutino      70.683333
                         Noturno       65.345000
                         Vespertino    69.783333
Mossoró                  Matutino      74.283333
                         Noturno       66.231579
                         Vespertino    73.778947
Natal-Cen

_Sua resposta escrita:_



## Questão 3 — `agg` com named aggregation (1,5 ponto)

Usando `groupby("campus")` e **named aggregation** (a forma
`nome=("coluna", "função")`), produza uma tabela com:

- `n_estudantes` — contagem de matrículas;
- `nota_media` — média de `nota_final`;
- `nota_desvio` — desvio padrão de `nota_final`;
- `taxa_aprovacao` — percentual de `situacao == "Aprovado"` (dica: uma
  função `lambda` que calcula a média de uma comparação booleana, vezes 100).

Ordene o resultado por `taxa_aprovacao`, do maior para o menor.

In [79]:
print("## agg com named aggregation \n")

resultado = df.groupby("campus").agg(
    n_estudantes=("matricula", "size"),
    nota_media=("nota_final", "mean"),
    nota_desvio=("nota_final", "std"),
    taxa_aprovacao=("situacao", lambda s: (s == "Aprovado").mean() * 100)
).sort_values("taxa_aprovacao", ascending=False).round(2)

print(resultado)

print("\n----------\n")


## agg com named aggregation 

                         n_estudantes  nota_media  nota_desvio  taxa_aprovacao
campus                                                                        
Parnamirim                         34       66.97        13.49           70.59
Apodi                              34       71.69        13.96           67.65
Natal-Zona Norte                   42       70.36        11.31           64.29
Santa Cruz                         25       74.55        12.51           64.00
Mossoró                            70       71.66        13.73           62.86
Caicó                              23       70.04         8.17           60.87
Currais Novos                      25       72.28        12.37           60.00
Natal-Central                      98       68.07        10.33           56.12
Macau                              34       67.18        11.10           55.88
Pau dos Ferros                     33       68.70        13.89           54.55
São Gonçalo do Amaran

_Sua resposta escrita:_



## Questão 4 — `pivot_table` (1,5 ponto)

a) Construa uma `pivot_table` com a nota média, `campus` nas linhas e
   `situacao` nas colunas, preenchendo células vazias com `0`
   (`fill_value=0`).
b) Refaça a mesma tabela, agora contando estudantes (em vez de média de
   nota), e adicione uma linha e uma coluna de **total**
   (`margins=True, margins_name="Total"`).
c) Em uma frase: por que faz mais sentido usar contagem, e não média, na
   pivot_table do item (b)?

In [80]:
print("## a) Construa uma pivot_table com a nota média, campus nas linhas e situacao nas colunas, preenchendo células vazias com 0 (fill_value=0)")
resultado = df.pivot_table(values="nota_final", aggfunc="mean", index="campus", columns="situacao", fill_value=0).round(2)
print(resultado)
print("\n----------\n")

print("## b) Refaça a mesma tabela, agora contando estudantes (em vez de média de nota), e adicione uma linha e uma coluna de total (margins=True, margins_name=\"Total\")")
resultado = df.pivot_table(values="matricula", aggfunc="count", index="campus", columns="situacao", fill_value=0, margins=True, margins_name="Total")
print(resultado)
print("\n----------\n")


## a) Construa uma pivot_table com a nota média, campus nas linhas e situacao nas colunas, preenchendo células vazias com 0 (fill_value=0)
situacao                 Aprovado  Reprovado por falta  Reprovado por nota
campus                                                                    
Apodi                       75.29                77.47               51.66
Caicó                       72.56                70.58               57.57
Currais Novos               78.03                73.50               54.06
João Câmara                 72.23                63.84               53.75
Macau                       73.17                63.60               52.37
Mossoró                     77.47                65.99               50.64
Natal-Central               71.43                69.19               52.26
Natal-Zona Norte            75.06                69.92               54.74
Parnamirim                  73.07                54.75               51.72
Pau dos Ferros              73.49   

_Sua resposta escrita:_

c) Em uma frase: por que faz mais sentido usar contagem, e não média, na pivot_table do item (b)?

Faz mais sentido usar contagem no item (b) porque está contando a quantidade de alunos por campus e usa como parametro o valor da matricula como agregador que é um texto (string) e dessa forma não faz sentido calcular a média.

## Questão 5 — `merge` muitos-para-um (1,5 ponto)

a) Combine `df` com `campi` pela coluna `campus`, com `how="left"` e
   `validate="m:1"`. Chame o resultado de `df_campi`.
b) Combine `df_campi` com `cursos` pela coluna `curso`, do mesmo jeito.
   Chame o resultado de `df_completo`.
c) Confirme que nenhuma linha ficou com `NaN` nas colunas vindas das
   tabelas `campi` ou `cursos` (dica: `.isna().sum()` nas colunas novas).
d) Por que `validate="m:1"` é uma boa prática aqui? O que ela teria
   acusado se `campi` tivesse duas linhas para o mesmo campus?

In [81]:
print("## a) Combine df com campi pela coluna campus, com how=\"left\" e validate=\"m:1\". Chame o resultado de df_campi \n")

# As variáveis abaixo foram criadas no início do arquivo e foram colocadas aqui apenas para verificação.
# df = pd.read_csv("/content/drive/MyDrive/aula05_laboratorio_dados/dados/matriculas_ifrn_2026.csv", sep=";", decimal=",",
#                 encoding="utf-8", dtype={"matricula": "string"})
# df["campus"] = df["campus"].str.strip()
# df = df.drop_duplicates().reset_index(drop=True)
# campi = pd.read_excel("/content/drive/MyDrive/aula05_laboratorio_dados/dados/campi_ifrn.xlsx", sheet_name="campi")
# cursos = pd.read_excel("/content/drive/MyDrive/aula05_laboratorio_dados/dados/campi_ifrn.xlsx", sheet_name="cursos")

df_campi = df.merge(campi, on="campus", how="left", validate="m:1")
print(df_campi.head(2))
print("\n----------\n")

print("## b) Combine df_campi com cursos pela coluna curso, do mesmo jeito. Chame o resultado de df_completo \n")
df_completo = df_campi.merge(cursos, on="curso", how="left", validate="m:1")
print(df_completo.head(2))
print("\n----------\n")

print("## c) Confirme que nenhuma linha ficou com NaN nas colunas vindas das tabelas campi ou cursos (dica: .isna().sum() nas colunas novas) \n")
print(df_completo[["campus", "cidade", "regiao", "ano_criacao", "num_docentes", "curso", "eixo_tecnologico", "nivel", "carga_horaria"]].isna().sum())
print("\n----------\n")

## a) Combine df com campi pela coluna campus, com how="left" e validate="m:1". Chame o resultado de df_campi 

  matricula                         nome campus  \
0  20261000  Wagner Cavalcante Fernandes  Apodi   
1  20261001        Zélia Santos Oliveira  Macau   

                                   curso  modalidade    turno  ano_ingresso  \
0  Análise e Desenvolvimento de Sistemas  Presencial  Noturno          2024   
1                  Redes de Computadores  Presencial  Noturno          2025   

   semestre  idade  nota_final  frequencia            situacao cidade  \
0         2     20        34.4        91.4  Reprovado por nota  Apodi   
1         1     24        70.9        79.3            Aprovado  Macau   

         regiao  ano_criacao  num_docentes  
0         Oeste         2009            88  
1  Costa Branca         2010            71  

----------

## b) Combine df_campi com cursos pela coluna curso, do mesmo jeito. Chame o resultado de df_completo 

  matricula             

_Sua resposta escrita:_

d) Por que validate="m:1" é uma boa prática aqui? O que ela teria acusado se campi tivesse duas linhas para o mesmo campus?

Aqui é uma boa prática usar validate="m:1" (relação de muitos para um) porque muitos estudantes pertencem a um campus. Esse parâmetro (validate) teria verifica que a tabela da direita campi tem no máximo uma linha por valor de chave evitando a duplicação.


## Questão 6 — `how="left"` x `how="inner"` (1,0 ponto)

Crie `cursos_parcial`, removendo de `cursos` as linhas cujo
`eixo_tecnologico` seja `"Informação e Comunicação"`. Em seguida:

a) Faça `df.merge(cursos_parcial, on="curso", how="inner")` e conte as linhas.
b) Faça `df.merge(cursos_parcial, on="curso", how="left")` e conte as linhas.
c) Explique, em uma frase, a diferença entre os dois resultados.

In [82]:
print("## Questão 6 — how=\"left\" x how=\"inner\"")
print("## Crie cursos_parcial, removendo de cursos as linhas cujo eixo_tecnologico seja \"Informação e Comunicação\" \n")
cursos_parcial = cursos[cursos["eixo_tecnologico"] != "Informação e Comunicação"]
print(cursos_parcial)
print("\n----------\n")

print("## a) Faça df.merge(cursos_parcial, on=\"curso\", how=\"inner\") e conte as linhas. \n")
resultado = df.merge(cursos_parcial, on="curso", how="inner")
print("Quantidade de linhas: ", len(resultado))
print("\n----------\n")

print("## b) Faça df.merge(cursos_parcial, on=\"curso\", how=\"left\") e conte as linhas. \n")
resultado = df.merge(cursos_parcial, on="curso", how="left")
print("Quantidade de linhas: ", len(resultado))
print("\n----------\n")

## Questão 6 — how="left" x how="inner"
## Crie cursos_parcial, removendo de cursos as linhas cujo eixo_tecnologico seja "Informação e Comunicação" 

           curso      eixo_tecnologico    nivel  carga_horaria
4    Edificações        Infraestrutura  Técnico           1300
5  Eletrotécnica  Controle e Processos  Técnico           1300
6       Mecânica  Controle e Processos  Técnico           1300
7        Química   Produção Industrial  Técnico           1200
8   Agroecologia     Recursos Naturais  Técnico           1200
9  Administração     Gestão e Negócios  Técnico           1200

----------

## a) Faça df.merge(cursos_parcial, on="curso", how="inner") e conte as linhas. 

Quantidade de linhas:  271

----------

## b) Faça df.merge(cursos_parcial, on="curso", how="left") e conte as linhas. 

Quantidade de linhas:  480

----------



_Sua resposta escrita:_

c) Explique, em uma frase, a diferença entre os dois resultados.

A diferença entre o resultados acontece porque quando usa how="inner" conta apenas as linhas que tem em comum entre "df" e "cursos" nas duas tabelas então não considera os eixo_tecnologico "Informação e Comunicação" que foram retiradas de "cursos", mas quando usa how="left" conta todas as linhas da tabela esquerda "df" inclusive com valores ausentes "NaN" indepente de ter retirado os eixo_tecnologico "Informação e Comunicação" da tabela "cursos".


## Questão 7 — `apply` com `lambda` (1,0 ponto)

a) Usando `df_completo["situacao"].apply(...)` com uma função `lambda`,
   crie a coluna `aprovado` (booleana: `True` se `situacao == "Aprovado"`).
b) Reescreva o mesmo cálculo **sem** `apply` (de forma vetorizada, em uma
   linha). Confirme que os dois resultados são idênticos.
c) Em uma frase: por que a versão vetorizada é preferível na maioria dos
   casos?

In [83]:
print("## Questão 7 — apply com lambda")
print("## a) Usando df_completo[\"situacao\"].apply(...) com uma função lambda, crie a coluna aprovado (booleana: True se situacao == \"Aprovado\") \n")
df_completo1 = df_completo
df_completo1["aprovado"] = df_completo1["situacao"].apply(lambda s: s == "Aprovado")
print(df_completo1.head(2))
print("\n----------\n")

print("## b) Reescreva o mesmo cálculo sem apply (de forma vetorizada, em uma linha). Confirme que os dois resultados são idênticos. \n")
df_completo2 = df_completo
df_completo2["aprovado"] = df_completo2["situacao"] == "Aprovado"
print(df_completo2.head(2))

print("\n### Confirme que os dois resultados de a) e b) são idênticos. \n")
print(df_completo1.equals(df_completo2))
print("\n----------\n")


## Questão 7 — apply com lambda
## a) Usando df_completo["situacao"].apply(...) com uma função lambda, crie a coluna aprovado (booleana: True se situacao == "Aprovado") 

  matricula                         nome campus  \
0  20261000  Wagner Cavalcante Fernandes  Apodi   
1  20261001        Zélia Santos Oliveira  Macau   

                                   curso  modalidade    turno  ano_ingresso  \
0  Análise e Desenvolvimento de Sistemas  Presencial  Noturno          2024   
1                  Redes de Computadores  Presencial  Noturno          2025   

   semestre  idade  nota_final  frequencia            situacao cidade  \
0         2     20        34.4        91.4  Reprovado por nota  Apodi   
1         1     24        70.9        79.3            Aprovado  Macau   

         regiao  ano_criacao  num_docentes          eixo_tecnologico  \
0         Oeste         2009            88  Informação e Comunicação   
1  Costa Branca         2010            71  Informação e Comunicação   



_Sua resposta escrita:_

c) Em uma frase: por que a versão vetorizada é preferível na maioria dos casos?

Vetorizada é preferível porque o processamento é mais rápido principalmente quando se tem milhares de linhas.


## Questão 8 — `apply(axis=1)` (1,0 ponto)

Escreva uma função `classificar_carga(linha)` que recebe uma linha de
`df_completo` e devolve:

- `"Curso longo"` se `carga_horaria >= 1800`;
- `"Curso médio"` se `1200 <= carga_horaria < 1800`;
- `"Curso curto"` caso contrário.

Aplique com `df_completo.apply(classificar_carga, axis=1)`, salve em uma
coluna `porte_curso` e mostre `value_counts()`.

In [84]:
print("Questão 8 — apply(axis=1) \n")
def classificar_carga(linha):
  if 1800 <= linha["carga_horaria"]:
    return "Curso longo"
  elif 1200 <= linha["carga_horaria"] and linha["carga_horaria"] < 1800:
    return "Curso médio"
  return "Curso curto"

df_completo["porte_curso"] = df_completo.apply(classificar_carga, axis=1)

print("\nPorte dos cursos:\n")
print(df_completo["porte_curso"].value_counts())
print("\n----------\n")

Questão 8 — apply(axis=1) 


Porte dos cursos:

porte_curso
Curso médio    380
Curso longo     58
Curso curto     42
Name: count, dtype: int64

----------



_Sua resposta escrita:_



## Questão 9 — Method chaining (1,5 ponto)

Em **uma única cadeia de métodos** (um só `resultado = (...)`, sem
variáveis intermediárias), a partir de `df_completo`:

1. descarte estudantes com `situacao == "Trancado"` (`query`);
2. crie a coluna booleana `aprovado` (`assign`);
3. agrupe por `regiao` e `eixo_tecnologico`;
4. calcule `n` (contagem) e `taxa_aprovacao` (média de `aprovado`) com
   named aggregation;
5. mantenha só grupos com `n >= 10` (`query`);
6. ordene por `taxa_aprovacao` decrescente.

Mostre as 5 primeiras linhas do resultado.

In [85]:
print("Questão 9 — Method chaining \n")
resultado = df_completo.query("situacao != 'Trancado'") \
            .assign(aprovado=lambda df: df["situacao"] == "Aprovado") \
            .groupby(["regiao", "eixo_tecnologico"]) \
            .agg(n=("aprovado", "size"), taxa_aprovacao=("aprovado", "mean")) \
            .query("n >= 10") \
            .sort_values("taxa_aprovacao", ascending=False)
print(resultado.head(5))
print("\n----------\n")

Questão 9 — Method chaining 

                                    n  taxa_aprovacao
regiao        eixo_tecnologico                       
Metropolitana Produção Industrial  12        0.916667
Oeste         Recursos Naturais    14        0.857143
              Gestão e Negócios    19        0.842105
Metropolitana Recursos Naturais    21        0.714286
Oeste         Produção Industrial  16        0.687500

----------



_Sua resposta escrita:_



## Questão 10 — Uma pergunta respondida com dados (1,0 ponto)

> **Estudantes de cursos com carga horária maior têm nota média mais
> alta?**

a) Use `pd.cut()` para criar uma coluna `carga_faixa` a partir de
   `carga_horaria`, com faixas `[0, 1200]`, `(1200, 1800]` e
   `(1800, 3000]`, rotuladas `"Até 1200h"`, `"1201–1800h"` e
   `"Acima de 1800h"`.
b) Agrupe por `carga_faixa` e calcule a nota média (use
   `observed=True` no `groupby` para evitar aviso).
c) Escreva, em uma célula de texto, o que os números mostram — e **uma
   limitação** dessa conclusão (pense em quais outras variáveis diferem
   entre cursos curtos e longos).

In [86]:
print("Questão 10 — Uma pergunta respondida com dados \n")
print("## a) Use pd.cut() para criar uma coluna carga_faixa a partir de carga_horaria, com faixas: \n")
df_completo["carga_faixa"] = pd.cut(df_completo["carga_horaria"],
                                    bins=[0, 1200, 1800, 3000],
                                    labels=["Até 1200h", "1201–1800h", "Acima de 1800h"])
print(df_completo.head(2))
print("\n----------\n")

print("## b) Agrupe por carga_faixa e calcule a nota média (use observed=True no groupby para evitar aviso). \n")
resultado = df_completo.groupby("carga_faixa", observed=True)["nota_final"].mean()
print(resultado.round(2))
print("\n----------\n")

Questão 10 — Uma pergunta respondida com dados 

## a) Use pd.cut() para criar uma coluna carga_faixa a partir de carga_horaria, com faixas: 

  matricula                         nome campus  \
0  20261000  Wagner Cavalcante Fernandes  Apodi   
1  20261001        Zélia Santos Oliveira  Macau   

                                   curso  modalidade    turno  ano_ingresso  \
0  Análise e Desenvolvimento de Sistemas  Presencial  Noturno          2024   
1                  Redes de Computadores  Presencial  Noturno          2025   

   semestre  idade  nota_final  ...  cidade        regiao ano_criacao  \
0         2     20        34.4  ...   Apodi         Oeste        2009   
1         1     24        70.9  ...   Macau  Costa Branca        2010   

  num_docentes          eixo_tecnologico     nivel carga_horaria aprovado  \
0           88  Informação e Comunicação  Superior          2400    False   
1           71  Informação e Comunicação   Técnico          1200     True   

   porte_curs

_Sua resposta escrita:_

c) Escreva, em uma célula de texto, o que os números mostram — e uma limitação dessa conclusão (pense em quais outras variáveis diferem entre cursos curtos e longos).

Os números mostram que quanto maior a carga horária (mais longo o curso) menor a média das notas, ou seja, a média das notas são inversamente proporcional ao tempo do curso. Porém essa conclusão é precipitada porque correlações simples não consideram outras variáveis.


---

## Entrega

- Arquivo: `aula05_SEUNOME.ipynb`, **executado do início ao fim**.
- Prazo e local: tarefa **"Exercício Prático — Aula 05"** no Moodle.
- Critérios: código correto e legível (60%), uso adequado de `groupby`/
  `agg`/`pivot_table`/`merge` (20%), interpretação escrita dos resultados
  (20%).
